In [1]:
import pykeen
import pandas as pd
from pykeen.pipeline import pipeline
from pykeen.datasets import get_dataset
from pykeen.datasets.analysis import (
    get_entity_count_df,
    get_entity_relation_co_occurrence_df,
    get_relation_cardinality_types_df,
    get_relation_count_df,
    get_relation_functionality_df,
)
from pykeen.models import TransE

# import tqdm as notebook_tqdm
# pip install ipywidgets ?

In [2]:
dataset = get_dataset(dataset="primekg")
# dataset = get_dataset(dataset="DRKG")


In [8]:
prime_kg = pd.read_csv("primekg/kg.csv", low_memory=False)


In [9]:
prime_kg.x_id.unique()

<StringArray>
[         '9796',          '7918',          '8233',          '4899',
          '5297',          '6564',          '8668',         '10826',
          '4489',          '6272',
 ...
  'R-HSA-975574',  'R-HSA-975577',  'R-HSA-975578',  'R-HSA-983168',
  'R-HSA-983170',  'R-HSA-983189', 'R-HSA-5690714',  'R-HSA-983695',
  'R-HSA-936837',  'R-HSA-997272']
Length: 90067, dtype: str

In [1]:
from dataset import load_dataset_config, PrimeKGDatasetBuilder
from pathlib import Path

# 1) Load separate dataset-only config
cfg = load_dataset_config(Path("dataset_config.yaml"))

# Optional quick experiment overrides in-memory
# cfg.runtime.max_rows = 200_000  # fast debug run, set to None for full data
cfg.filters.relation_whitelist = []  # fill with relation names to filter
cfg.export.enabled = False  # True to persist snapshot

# 2) Build dataset artifacts in-memory
artifacts = PrimeKGDatasetBuilder(cfg).build()

# 3) Access splits + feature registry
print("Triples:", artifacts.training.num_triples, artifacts.validation.num_triples, artifacts.testing.num_triples)
print("Feature stats:", artifacts.feature_registry.stats)
print("Dataset stats:", artifacts.stats)

# Even shorter: artifacts = PrimeKGDatasetBuilder().build()

c:\diploma\Disease-Gene-Prioritization-Project\dataset.py:193: DtypeWarning: Columns (0: x_id, 1: y_id) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(triples_path, **read_kwargs)


Dataset build summary
- triples: 8100498 -> 867360 (dropped: 7233138)
- relations: 3
- entities: 36098
- split sizes: train=780624, validation=43368, test=43368
Triples: 780624 43368 43368
Feature stats: {'enabled': True, 'entity_count': 36098, 'metadata_coverage': 36098, 'text_coverage': 0}
Dataset stats: {'triples_before_filtering': 8100498, 'triples_after_filtering': 867360, 'unique_relations': 3, 'unique_entities_in_triples': 36098, 'total_nodes_loaded': 129375, 'split': {'train': 0.9, 'validation': 0.05, 'test': 0.05, 'random_seed': 42}, 'records_dropped': 7233138}


In [2]:
# Betöltött relációk (a teljes, szűrt datasetből)
relations = sorted(artifacts.all_triples.relation_to_id.keys())
print(f"Relation count: {len(relations)}")
print(relations[:30], "..." if len(relations) > 30 else "")

# Betöltött node típusok (feature registry-ből)
node_meta = artifacts.feature_registry.entity_metadata
if not node_meta:
    print("Nincs node metadata. Kapcsold be: cfg.features.include_node_metadata = True")
else:
    node_types = sorted({m.get('node_type') for m in node_meta.values() if m.get('node_type')})
    print(f"Node type count: {len(node_types)}")
    print(node_types)

    type_counts = {}
    for m in node_meta.values():
        t = m.get('node_type')
        if t:
            type_counts[t] = type_counts.get(t, 0) + 1
    print("\nNode counts by type:")
    for t, c in sorted(type_counts.items(), key=lambda x: (-x[1], x[0])):
        print(f"- {t}: {c}")

Relation count: 3
['disease_disease', 'disease_protein', 'protein_protein'] 
Node type count: 2
['disease', 'gene/protein']

Node counts by type:
- gene/protein: 19054
- disease: 17044


In [2]:
from pykeen.pipeline import pipeline

result = pipeline(
    training=artifacts.training,
    validation=artifacts.validation,
    testing=artifacts.testing,
    model='DistMult',
    training_kwargs=dict(num_epochs=10),
)

No random seed is specified. Setting to 10761983.


Training epochs on cuda:0:   0%|          | 0/10 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/3.05k [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/3.05k [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/3.05k [00:00<?, ?batch/s]

...

Training batches on cuda:0:   0%|          | 0.00/3.05k [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/3.05k [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/3.05k [00:00<?, ?batch/s]

Evaluating on cuda:0:   0%|          | 0.00/43.4k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 225.79s seconds


In [3]:
print(result.get_metric('hits@10'))
print(result.get_metric('mean_reciprocal_rank'))

0.035925106068990964
0.027306117117404938


In [ ]:
result.losses()

In [ ]:
result.plot_early_stopping()


In [4]:
artifacts.feature_registry.text_features

{}

In [5]:
import pandas as pd

features_df = pd.read_csv('primekg/disease_features.csv')

In [11]:
features_df.head()

,node_index,mondo_id,mondo_name,group_id_bert,group_name_bert,mondo_definition,umls_description,orphanet_definition,orphanet_prevalence,orphanet_epidemiology,orphanet_clinical_description,orphanet_management_and_treatment,mayo_symptoms,mayo_causes,mayo_risk_factors,mayo_complications,mayo_prevention,mayo_see_doc
0,27165,8019,mullerian aplasia and hyperandrogenism,NaN,NaN,"Deficiency of the glycoprotein WNT4, associate...","Deficiency of the glycoprotein wnt4, associate...","A rare syndrome with 46,XX disorder of sex dev...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,27165,8019,mullerian aplasia and hyperandrogenism,NaN,NaN,"Deficiency of the glycoprotein WNT4, associate...","Deficiency of the glycoprotein wnt4, associate...","A rare syndrome with 46,XX disorder of sex dev...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,27166,11043,"myelodysplasia, immunodeficiency, facial dysmo...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,27168,8878,"bone dysplasia, lethal Holmgren type",NaN,NaN,Bone dysplasia lethal Holmgren type (BDLH) is ...,A lethal bone dysplasia with characteristics o...,Bone dysplasia lethal Holmgren type (BDLH) is ...,<1/1000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,27169,8905,predisposition to invasive fungal disease due ...,NaN,NaN,NaN,NaN,"A rare, genetic primary immunodeficiency chara...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
